# Forecasting Tutorial Notebook
This notebook teaches a complete forecasting workflow:
1. Define the forecasting objective
2. Prepare a time series dataset
3. Build a baseline and a forecasting model
4. Validate with time-aware splits
5. Generate and interpret predictions
6. Prepare the notebook to share on MyBinder

## How to Use This Notebook (Beginner Version)
This notebook is designed for people with little or no forecasting experience.
You only need to run cells from top to bottom.

### What to do
1. Run Cell 3 (import required libraries).
2. Run Cells 5 and 7 to see the basic example with synthetic data.
3. In the activity section, run Cell 9 to load data (you can keep synthetic).
4. Edit values in Cell 10 only if you want to explore the capabilities.
5. Run Cell 10 to train and validate the model.
6. Run Cell 11 to predict future months.

### What you can edit safely
- In Cell 9: choose data source (`synthetic`, `csv_url`, `csv_file`, or `sql`).
- In Cell 10:
  - `prediction_window`: how many last points are kept for final testing.
  - `lags`: which past months the model uses.
  - `n_splits`: how many validation rounds to run.
  - `forecast_periods`: how many future months to predict.

### If something goes wrong
- Do not panic: restart the kernel and run cells again from Cell 3 downward.
- Keep `source_type='synthetic'` first, then try external data later.

### How to read results
- Lower MAE, RMSE, and MAPE means better predictions.
- Compare model values with naive values: if model is lower, it is better than the simple baseline.

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import TimeSeriesSplit

# Reproducibility
np.random.seed(42)

# Train/evaluate with chosen parameters and prediction window
from sklearn.ensemble import RandomForestRegressor

def train_evaluate_with_params(df_input, activity_params):
    params = activity_params.copy()
    model_type = params['model_type']
    prediction_window = int(params['prediction_window'])
    lags = list(params['lags'])
    n_splits = int(params['n_splits'])
    rf_estimators = int(params['random_forest_estimators'])
    random_state = int(params['random_state'])

    if prediction_window < 1:
        raise ValueError('prediction_window must be >= 1')
    if len(df_input) <= max(lags) + prediction_window:
        raise ValueError('Not enough data for these lags and prediction_window values.')

    activity_df = df_input[['y']].copy()
    for lag in lags:
        activity_df[f'lag_{lag}'] = activity_df['y'].shift(lag)

    activity_df = activity_df.dropna().copy()
    X_all = activity_df[[f'lag_{lag}' for lag in lags]]
    y_all = activity_df['y']

    X_train = X_all.iloc[:-prediction_window]
    y_train = y_all.iloc[:-prediction_window]
    X_test = X_all.iloc[-prediction_window:]
    y_test = y_all.iloc[-prediction_window:]

    if len(X_train) < n_splits + 1:
        raise ValueError('n_splits is too large for the available training data.')

    def build_model():
        if model_type == 'linear':
            return LinearRegression()
        if model_type == 'random_forest':
            return RandomForestRegressor(
                n_estimators=rf_estimators,
                random_state=random_state,
            )
        raise ValueError("model_type must be 'linear' or 'random_forest'")

    def mape(y_true, y_pred):
        y_true_arr = np.asarray(y_true)
        y_pred_arr = np.asarray(y_pred)
        safe = np.where(y_true_arr == 0, np.nan, y_true_arr)
        return np.nanmean(np.abs((y_true_arr - y_pred_arr) / safe)) * 100

    tscv = TimeSeriesSplit(n_splits=n_splits)
    cv_rows = []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train), start=1):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = build_model()
        model.fit(X_tr, y_tr)
        val_pred = model.predict(X_val)
        naive_val_pred = X_val['lag_1'].values

        cv_rows.append({
            'fold': fold,
            'model_mae': mean_absolute_error(y_val, val_pred),
            'model_rmse': mean_squared_error(y_val, val_pred) ** 0.5,
            'model_mape': mape(y_val, val_pred),
            'naive_mae': mean_absolute_error(y_val, naive_val_pred),
            'naive_rmse': mean_squared_error(y_val, naive_val_pred) ** 0.5,
            'naive_mape': mape(y_val, naive_val_pred),
        })

    cv_results = pd.DataFrame(cv_rows)
    display(cv_results.round(2))

    print('Average CV metrics (lower is better):')
    display(cv_results.drop(columns=['fold']).mean().to_frame('mean').round(2))

    final_activity_model = build_model()
    final_activity_model.fit(X_train, y_train)
    test_pred = final_activity_model.predict(X_test)
    naive_test_pred = X_test['lag_1'].values

    holdout_metrics = pd.DataFrame([
        {
            'model_type': model_type,
            'prediction_window': prediction_window,
            'model_mae': mean_absolute_error(y_test, test_pred),
            'model_rmse': mean_squared_error(y_test, test_pred) ** 0.5,
            'model_mape': mape(y_test, test_pred),
            'naive_mae': mean_absolute_error(y_test, naive_test_pred),
            'naive_rmse': mean_squared_error(y_test, naive_test_pred) ** 0.5,
            'naive_mape': mape(y_test, naive_test_pred),
        }
    ])

    print('Holdout-window performance (last points of the series):')
    display(holdout_metrics.round(2))

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(y_test.index, y_test.values, marker='o', label='Observed')
    ax.plot(y_test.index, test_pred, marker='o', label=f'Model ({model_type})')
    ax.plot(y_test.index, naive_test_pred, marker='o', linestyle='--', label='Naive baseline (lag_1)')
    ax.set_title(f'Holdout Forecast Comparison | window={prediction_window}')
    ax.set_ylabel('Target')
    ax.legend()
    plt.show()

    return {
        'cv_results': cv_results,
        'holdout_metrics': holdout_metrics,
        'predictions': pd.DataFrame({
            'observed': y_test.values,
            'model_pred': test_pred,
            'naive_pred': naive_test_pred,
        }, index=y_test.index),
    }

## Step 1: Create a Synthetic Time Series (Guided Example)
In this first model, we always use synthetic data to learn the forecasting workflow clearly.
Later, in the activity section, you can switch to external data sources (CSV/URL/SQL).

In [ ]:
# Generate 60 months of synthetic data (guided example)
dates = pd.date_range(start='2019-01-01', periods=60, freq='MS')
t = np.arange(len(dates))

trend = 2.0 * t
seasonality = 12 * np.sin(2 * np.pi * t / 12)
noise = np.random.normal(loc=0, scale=4, size=len(t))

y = 100 + trend + seasonality + noise

df = pd.DataFrame({'date': dates, 'y': y})
df = df.set_index('date')

# Plot
ax = df['y'].plot(figsize=(12, 4), title='Synthetic Monthly Demand')
ax.set_ylabel('Demand')
plt.show()

# Build lag features (predict current value from previous months)
for lag in [1, 2, 3, 6, 12]:
    df[f'lag_{lag}'] = df['y'].shift(lag)

df_model = df.dropna().copy()
df_model.head()

## Step 2: Train and Validate the Model
We validate using `TimeSeriesSplit`, which respects chronological order.
This avoids data leakage and gives a realistic estimate of forecasting quality.

In [ ]:
# Features and target
feature_cols = [c for c in df_model.columns if c.startswith('lag_')]
X = df_model[feature_cols]
y = df_model['y']

# Time-aware cross-validation
tscv = TimeSeriesSplit(n_splits=5)

mae_scores = []
rmse_scores = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred) ** 0.5
    mae_scores.append(mae)
    rmse_scores.append(rmse)

    print(f'Fold {fold}: MAE={mae:.2f}, RMSE={rmse:.2f}')

print('\nAverage validation MAE:', round(np.mean(mae_scores), 2))
print('Average validation RMSE:', round(np.mean(rmse_scores), 2))

# Final model on all available data
final_model = LinearRegression()
final_model.fit(X, y)

# One-step-ahead example prediction (next month)
last_known = df['y'].copy()
next_features = pd.DataFrame({
    'lag_1': [last_known.iloc[-1]],
    'lag_2': [last_known.iloc[-2]],
    'lag_3': [last_known.iloc[-3]],
    'lag_6': [last_known.iloc[-6]],
    'lag_12': [last_known.iloc[-12]],
})

next_pred = final_model.predict(next_features)[0]
next_date = df.index[-1] + pd.offsets.MonthBegin(1)

print(f'\nForecast for {next_date.date()}: {next_pred:.2f}')

# Visualize recent history and the forecast point
plot_df = df.tail(24).copy()
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(plot_df.index, plot_df['y'], label='Observed')
ax.scatter([next_date], [next_pred], color='red', label='Forecast (t+1)', zorder=5)
ax.set_title('Recent History and One-Step Forecast')
ax.set_ylabel('Demand')
ax.legend()
plt.show()


## Activity: Step 1A (Optional External Data) + Model Comparison
In this activity, you can keep synthetic data or load a new dataset from CSV URL, local CSV, or SQL.
Then tune model parameters and compare performance against the naive baseline.

Questions to answer:
1. How does performance change when you increase the prediction window?
2. Which metric changes the most (MAE, RMSE, MAPE)? Why might that happen?
3. Compare `linear` and `random_forest`. Which one generalizes better here?
4. Does the model perform equally well across all points in the horizon?
5. What would you change in the features to improve long-horizon forecasts?

In [ ]:

 # Data loading configuration
activity_data_config = {
    # Options: 'synthetic', 'csv_url', 'csv_file', 'sql'
    'source_type': 'synthetic',

    # External data schema
    'date_col': 'date',
    'target_col': 'y',

    # Public CSV URL option
    'csv_url': '',

    # CSV file path (inside Binder session/repo)
    'csv_file_path': '',

    # SQL template (do not hardcode secrets in notebooks)
    # Example SQLite URI: 'sqlite:///my_data.db'
    'sql_connection_uri': '',
    'sql_query': 'SELECT date, y FROM your_table ORDER BY date',
}

# Step 1A (Activity): choose and load data
def build_synthetic_data(periods=60):
    dates = pd.date_range(start='2019-01-01', periods=periods, freq='MS')
    t = np.arange(len(dates))
    trend = 2.0 * t
    seasonality = 12 * np.sin(2 * np.pi * t / 12)
    noise = np.random.normal(loc=0, scale=4, size=len(t))
    y = 100 + trend + seasonality + noise
    return pd.DataFrame({'date': dates, 'y': y})

def load_time_series_data(config):
    source_type = config.get('source_type', 'synthetic')
    date_col = config.get('date_col', 'date')
    target_col = config.get('target_col', 'y')

    if source_type == 'synthetic':
        raw_df = build_synthetic_data()
    elif source_type == 'csv_url':
        if not config.get('csv_url'):
            raise ValueError('Set activity_data_config[\'csv_url\'] for source_type=\'csv_url\'.')
        raw_df = pd.read_csv(config['csv_url'])
    elif source_type == 'csv_file':
        if not config.get('csv_file_path'):
            raise ValueError('Set activity_data_config[\'csv_file_path\'] for source_type=\'csv_file\'.')
        raw_df = pd.read_csv(config['csv_file_path'])
    elif source_type == 'sql':
        if not config.get('sql_connection_uri'):
            raise ValueError('Set activity_data_config[\'sql_connection_uri\'] for source_type=\'sql\'.')
        if not config.get('sql_query'):
            raise ValueError('Set activity_data_config[\'sql_query\'] for source_type=\'sql\'.')

        try:
            from sqlalchemy import create_engine
        except ImportError as exc:
            raise ImportError('Install sqlalchemy to use SQL loading.') from exc

        engine = create_engine(config['sql_connection_uri'])
        raw_df = pd.read_sql_query(config['sql_query'], con=engine)
    else:
        raise ValueError("source_type must be one of: 'synthetic', 'csv_url', 'csv_file', 'sql'.")

    if date_col not in raw_df.columns or target_col not in raw_df.columns:
        raise ValueError(
            f"Input data must include columns '{date_col}' and '{target_col}'. "
            f'Available columns: {list(raw_df.columns)}'
        )

    loaded = raw_df[[date_col, target_col]].copy()
    loaded[date_col] = pd.to_datetime(loaded[date_col], errors='coerce')
    loaded = loaded.dropna(subset=[date_col, target_col]).sort_values(date_col)
    loaded = loaded.rename(columns={target_col: 'y'}).set_index(date_col)

    if not loaded.index.is_unique:
        loaded = loaded.groupby(level=0).mean()

    return loaded

activity_df_input = load_time_series_data(activity_data_config)
print(f"Loaded source: {activity_data_config['source_type']}")
print(f'Shape: {activity_df_input.shape}')
display(activity_df_input.head())

# Plot generated/read data
if activity_df_input.empty:
    raise ValueError('Loaded dataset is empty. Please check your data source settings.')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(activity_df_input.index, activity_df_input['y'], marker='o', linewidth=1.5)
ax.set_title(f"Loaded Time Series ({activity_data_config['source_type']})")
ax.set_ylabel('Target')
ax.set_xlabel('Date')
plt.show()


In [ ]:
# Editable model parameters
activity_params = {
    'model_type': 'linear',        # Options: 'linear', 'random_forest'
    'prediction_window': 3,       # Number of last periods reserved for test
    'lags': [1, 2, 3, 6, 12],      # Lag features to create
    'n_splits': 5,                 # TimeSeriesSplit folds on training subset
    'random_forest_estimators': 200,
    'random_state': 2432,
}

# Editable prediction horizon (months ahead)
forecast_periods = 6

activity_params
print(f'Forecast horizon (months): {forecast_periods}')

# Run model directly after setting parameters
activity_outputs = train_evaluate_with_params(activity_df_input, activity_params)

## Future Forecast (Editable Number of Periods)
This section generates recursive multi-step predictions using the selected activity data and parameters.
Set `forecast_periods` in the previous code cell to choose how many future months to predict.


In [ ]:
# Multi-step forecast for future periods
from pandas.tseries.frequencies import to_offset

def forecast_next_periods(df_input, activity_params, n_future_periods=12):
    lags = list(activity_params['lags'])
    model_type = activity_params['model_type']
    rf_estimators = int(activity_params['random_forest_estimators'])
    random_state = int(activity_params['random_state'])

    data = df_input[['y']].copy().sort_index()
    for lag in lags:
        data[f'lag_{lag}'] = data['y'].shift(lag)

    train_df = data.dropna().copy()
    X_train = train_df[[f'lag_{lag}' for lag in lags]]
    y_train = train_df['y']

    if model_type == 'linear':
        model = LinearRegression()
    elif model_type == 'random_forest':
        model = RandomForestRegressor(
            n_estimators=rf_estimators,
            random_state=random_state,
        )
    else:
        raise ValueError("model_type must be 'linear' or 'random_forest'")

    model.fit(X_train, y_train)

    inferred_freq = pd.infer_freq(data.index)
    if inferred_freq is None:
        inferred_freq = 'MS'
    step_offset = to_offset(inferred_freq)

    history = data['y'].copy()
    future_rows = []

    for _ in range(n_future_periods):
        next_x = pd.DataFrame([{f'lag_{lag}': history.iloc[-lag] for lag in lags}])
        next_y = float(model.predict(next_x)[0])
        next_date = history.index[-1] + step_offset

        future_rows.append({'date': next_date, 'forecast': next_y})
        history.loc[next_date] = next_y

    future_df = pd.DataFrame(future_rows).set_index('date')
    return future_df

if 'forecast_periods' not in globals():
    forecast_periods = 12

next_forecast = forecast_next_periods(
    activity_df_input,
    activity_params,
    n_future_periods=int(forecast_periods),
)

print(f'Next {forecast_periods}-period forecast:')
display(next_forecast.round(2))

fig, ax = plt.subplots(figsize=(12, 4))
history_to_plot = activity_df_input['y'].tail(24)
ax.plot(history_to_plot.index, history_to_plot.values, marker='o', label='Observed (last 24)')
ax.plot(next_forecast.index, next_forecast['forecast'].values, marker='o', label=f'Forecast (next {forecast_periods})')
ax.set_title(f'Forecast for Next {forecast_periods} Periods')
ax.set_ylabel('Target')
ax.legend()
plt.show()

## How to Interpret the Results
Use this guide after running the activity cell:

### Metric Definitions
1. **Naive baseline**
   - A simple reference model that predicts the current value using the previous value (`lag_1`).
   - It is a minimum benchmark: your model should ideally outperform this baseline.

2. **MAE (Mean Absolute Error)**
   - Average absolute difference between observed and predicted values.
   - Formula: $\mathrm{MAE}=\frac{1}{n}\sum_{i=1}^{n}|y_i-\hat{y}_i|$
   - Same units as the target variable (here, demand).
   - Easy to interpret and less sensitive to outliers than RMSE.

3. **RMSE (Root Mean Squared Error)**
   - Square root of the average squared error.
   - Formula: $\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_{i=1}^{n}(y_i-\hat{y}_i)^2}$
   - Same units as the target variable.
   - Penalizes large errors more strongly than MAE, so it highlights occasional big misses.

4. **MAPE (Mean Absolute Percentage Error)**
   - Average absolute percentage error.
   - Formula: $\mathrm{MAPE}=\frac{100}{n}\sum_{i=1}^{n}\left|\frac{y_i-\hat{y}_i}{y_i}\right|$
   - Expressed as a percentage, useful for comparing errors across different scales.
   - Be careful when actual values are near zero, because percentages can become unstable.

### How to Read the Output
1. **Cross-validation table (`cv_results`)**
   - Each row is one time-based validation fold.
   - Compare `model_*` vs `naive_*` metrics.
   - Lower values are better for MAE, RMSE, and MAPE.
   - If the model is consistently lower than naive across folds, it is learning useful signal.

2. **Average CV metrics**
   - These summarize stability across folds.
   - Large differences between folds suggest the model is sensitive to time periods (possible regime changes).

3. **Holdout-window metrics (`holdout_metrics`)**
   - This is the most realistic performance estimate for future unseen points.
   - If holdout error is much worse than CV error, the model may be overfitting or the recent pattern changed.

4. **Forecast plot (Observed vs Model vs Naive)**
   - Check if the model follows turning points and trend direction.
   - If the model curve is too smooth or delayed, lag features may be insufficient.
   - If naive is similar or better, model complexity is not adding value for this dataset.

5. **Teaching interpretation prompts**
   - How does performance change when `prediction_window` increases?
   - Does `random_forest` improve holdout metrics or only CV metrics?
   - Which metric is most sensitive in your experiment, and why?